# Plant Disease Detection - ResNet9 (Colab GPU)

Backup training path for the ASEP project, for when a local CUDA install is not
available. Produces the same checkpoint format the local project expects, so the
Flask app and `src/evaluate.py` can consume it directly.

**Prefer training locally if you have an NVIDIA GPU** - no session limits, no
re-uploading 2.7 GB. Use this when you do not.

Steps: set the runtime to GPU (*Runtime > Change runtime type > T4 GPU*), run every
cell, then download `plant_disease_resnet9.pth` and `class_names.json` into the
project's `models/` folder.


## 1. Confirm the GPU

In [ ]:
import torch, subprocess
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, "| VRAM: %.1f GB" % (p.total_memory / 1e9))
else:
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then rerun.")


## 2. Get the dataset

Upload your Kaggle API token (`kaggle.json`, from kaggle.com > Settings > API >
Create New Token). Downloading inside Colab takes ~2 minutes versus hours of
local unzipping, because it is a datacentre SSD with no antivirus in the path.


In [ ]:
from google.colab import files
print("Upload kaggle.json:")
files.upload()

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content


### Extract only what is needed

The archive contains the dataset **twice** in two folders whose names differ only
in case. On Linux both extract fine but you pay double the time and disk; we skip
the duplicate and flatten the doubled nesting.


In [ ]:
import zipfile, time, os
from concurrent.futures import ThreadPoolExecutor

ZIP = "/content/new-plant-diseases-dataset.zip"
ROOT = "/content/data"
CANON = "New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/"

zf = zipfile.ZipFile(ZIP)
infos = [i for i in zf.infolist() if not i.is_dir()]
jobs = []
for i in infos:
    n = i.filename
    if n.startswith(CANON):
        rel = n[len(CANON):]
        parts = rel.split("/")
        if len(parts) >= 3 and parts[0] in ("train", "valid"):
            jobs.append((n, f"{parts[0]}/{parts[1]}/{parts[-1]}"))
    elif n.startswith("test/"):
        jobs.append((n, "test/" + os.path.basename(n)))

print(f"{len(jobs):,} files to extract ({len(infos) - len(jobs):,} duplicates skipped)")
for d in {os.path.dirname(o) for _, o in jobs}:
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)

import threading
_local = threading.local()
def handle():
    z = getattr(_local, "z", None)
    if z is None:
        z = zipfile.ZipFile(ZIP); _local.z = z
    return z

def work(job):
    name, rel = job
    with handle().open(name) as src, open(os.path.join(ROOT, rel), "wb") as fh:
        fh.write(src.read())

t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(work, jobs))
print("extracted in %.1f min" % ((time.time() - t0) / 60))

for split in ("train", "valid"):
    d = os.path.join(ROOT, split)
    n = sum(len(f) for _, _, f in os.walk(d))
    print(f"  {split}: {len(os.listdir(d))} classes, {n} images")


## 3. Model

Same architecture as `src/model.py`.

In [ ]:
import torch.nn as nn, torch.nn.functional as F

def conv_block(cin, cout, pool=False):
    layers = [nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
    if pool:
        layers.append(nn.MaxPool2d(4))
    return nn.Sequential(*layers)

def accuracy(out, labels):
    _, preds = torch.max(out, dim=1)
    return torch.tensor(torch.sum(preds == labels).item() / len(preds))

class ResNet9(nn.Module):
    """AdaptiveMaxPool2d(1) instead of a fixed MaxPool2d(4): identical at 256px,
    but also valid at 64/128px so the checkpoint works on a CPU machine too."""
    def __init__(self, in_channels=3, num_classes=38):
        super().__init__()
        self.conv1 = conv_block(in_channels, 64)
        self.conv2 = conv_block(64, 128, pool=True)
        self.res1 = nn.Sequential(conv_block(128, 128), conv_block(128, 128))
        self.conv3 = conv_block(128, 256, pool=True)
        self.conv4 = conv_block(256, 512, pool=True)
        self.res2 = nn.Sequential(conv_block(512, 512), conv_block(512, 512))
        self.classifier = nn.Sequential(nn.AdaptiveMaxPool2d(1), nn.Flatten(),
                                        nn.Linear(512, num_classes))

    def forward(self, x):
        out = self.conv1(x); out = self.conv2(out); out = self.res1(out) + out
        out = self.conv3(out); out = self.conv4(out); out = self.res2(out) + out
        return self.classifier(out)

    def training_step(self, batch):
        images, labels = batch
        return F.cross_entropy(self(images), labels)

    @torch.no_grad()
    def validation_step(self, batch):
        images, labels = batch
        out = self(images)
        return {"val_loss": F.cross_entropy(out, labels).detach(),
                "val_acc": accuracy(out, labels)}


## 4. Data loaders

Normalisation is `ToTensor()` only -> [0, 1], matching the paper and the local code.

In [ ]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as T

IMG_SIZE = 256
BATCH = 64

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), antialias=True),
    T.ToTensor(),
])
valid_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])

train_ds = ImageFolder(f"{ROOT}/train", train_tf)
valid_ds = ImageFolder(f"{ROOT}/valid", valid_tf)
class_names = train_ds.classes

# A final batch of exactly 1 sample crashes BatchNorm; drop it when that happens.
drop_last = len(train_ds) > BATCH and len(train_ds) % BATCH == 1

train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=2,
                      pin_memory=True, drop_last=drop_last)
valid_dl = DataLoader(valid_ds, BATCH * 2, num_workers=2, pin_memory=True)
print(len(train_ds), "train /", len(valid_ds), "valid /", len(class_names), "classes")


In [ ]:
def to_device(data, device):
    if isinstance(data, (list, tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

class DeviceDataLoader:
    def __init__(self, dl, device): self.dl, self.device = dl, device
    def __iter__(self):
        for b in self.dl: yield to_device(b, self.device)
    def __len__(self): return len(self.dl)

device = torch.device("cuda")
train_dl = DeviceDataLoader(train_dl, device)
valid_dl = DeviceDataLoader(valid_dl, device)
model = ResNet9(3, len(class_names)).to(device)
print("parameters: %.2fM" % (sum(p.numel() for p in model.parameters()) / 1e6))


## 5. Train

Adam + One-Cycle LR + gradient clipping + weight decay, as the paper specifies,
plus mixed precision for speed. ResNet9 converges by epoch 2-3 here; 5 is ample.


In [ ]:
import time
from tqdm.auto import tqdm

EPOCHS, MAX_LR, GRAD_CLIP, WEIGHT_DECAY = 5, 0.01, 0.1, 1e-4

@torch.no_grad()
def evaluate(model, dl):
    model.eval()
    outs = [model.validation_step(b) for b in dl]
    return {"val_loss": torch.stack([o["val_loss"] for o in outs]).mean().item(),
            "val_acc": torch.stack([o["val_acc"] for o in outs]).mean().item()}

opt = torch.optim.Adam(model.parameters(), MAX_LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, MAX_LR, epochs=EPOCHS,
                                            steps_per_epoch=len(train_dl))
scaler = torch.amp.GradScaler("cuda")

history, best_acc = [], -1.0
for epoch in range(EPOCHS):
    t0 = time.time(); model.train(); losses = []
    for batch in tqdm(train_dl, desc=f"epoch {epoch+1}/{EPOCHS}"):
        with torch.amp.autocast("cuda"):
            loss = model.training_step(batch)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_value_(model.parameters(), GRAD_CLIP)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        losses.append(loss.detach()); sched.step()

    res = evaluate(model, valid_dl)
    res.update(train_loss=torch.stack(losses).mean().item(),
               epoch=epoch + 1, secs=time.time() - t0)
    history.append(res)
    print("  epoch %d  train_loss=%.4f  val_loss=%.4f  val_acc=%.4f  (%.1f min)"
          % (res["epoch"], res["train_loss"], res["val_loss"], res["val_acc"],
             res["secs"] / 60))

    if res["val_acc"] > best_acc:
        best_acc = res["val_acc"]
        torch.save({"model": "resnet9", "state_dict": model.state_dict(),
                    "class_names": class_names, "img_size": IMG_SIZE,
                    "val_acc": best_acc, "num_classes": len(class_names),
                    "history": history,
                    "args": {"img_size": IMG_SIZE, "epochs": EPOCHS,
                             "batch_size": BATCH, "max_lr": MAX_LR, "amp": True}},
                   "plant_disease_resnet9.pth")
        print("    -> checkpoint saved (best so far)")

print("\nbest val_acc: %.4f (%.2f%%)" % (best_acc, best_acc * 100))


## 6. Curves

In [ ]:
import matplotlib.pyplot as plt, json

epochs = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(epochs, [h["val_acc"] * 100 for h in history], color="#2a78d6", marker="o")
axes[0].set_title("Validation accuracy vs epochs", loc="left")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy (%)")
axes[1].plot(epochs, [h["train_loss"] for h in history], color="#2a78d6", marker="o", label="Train")
axes[1].plot(epochs, [h["val_loss"] for h in history], color="#eb6834", marker="s", label="Validation")
axes[1].set_title("Loss vs epochs", loc="left")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Cross-entropy"); axes[1].legend(frameon=False)
for ax in axes:
    ax.set_xticks(epochs); ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e4e3df", linewidth=0.7)
plt.tight_layout(); plt.show()

with open("class_names.json", "w") as fh:
    json.dump(class_names, fh, indent=2)


## 7. Download

Put both files in the project's `models/` folder, then run locally:

```bash
python -m src.evaluate
python app/app.py
```


In [ ]:
from google.colab import files
files.download("plant_disease_resnet9.pth")
files.download("class_names.json")
